# Sentinel-2 RGB Composite with Date Selection

This notebook creates an RGB composite from Sentinel-2 imagery with interactive date range selection.

**Instructions:**
1. Run all cells to initialize Earth Engine and display the map
2. Draw a polygon on the map to define your area of interest
3. Set your start and end dates in the date widgets
4. Run the analysis cell to generate the composite
5. Export the result to ArcGIS Online or save locally

In [ ]:
# Import required libraries
import ee
import geemap
from datetime import datetime, timedelta
from ipywidgets import widgets, Layout
from IPython.display import display
import os

In [ ]:
# Initialize Earth Engine
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

In [ ]:
# Cloud masking function
def mask_s2_clouds(image):
    """
    Masks clouds and cirrus from Sentinel-2 imagery using QA60 band.
    
    Args:
        image: Sentinel-2 image
    
    Returns:
        Masked image with values scaled to 0-1
    """
    qa = image.select('QA60')
    
    # Bits 10 and 11 are clouds and cirrus, respectively
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    
    # Both flags should be set to zero, indicating clear conditions
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    
    return image.updateMask(mask).divide(10000)

In [ ]:
# Create the map centered on Quinhagak, Alaska
Map = geemap.Map(center=[59.750359237437635, -161.90402886481778], zoom=10)
Map.add_basemap('SATELLITE')

# Add drawing tools
Map.add_draw_control()

print("Map initialized. Use the drawing tools on the left to draw a polygon.")
Map

## Set Date Range

Select the start and end dates for your Sentinel-2 imagery composite.

In [ ]:
# Create date picker widgets
start_date_picker = widgets.DatePicker(
    description='Start Date:',
    value=datetime(2024, 5, 1),
    style={'description_width': '100px'},
    layout=Layout(width='300px')
)

end_date_picker = widgets.DatePicker(
    description='End Date:',
    value=datetime(2024, 10, 30),
    style={'description_width': '100px'},
    layout=Layout(width='300px')
)

# Status output
status_output = widgets.Output()

# Display date pickers
display(start_date_picker, end_date_picker)

## Run Analysis

Execute this cell to process the imagery and display the composite on the map.

In [ ]:
# Main analysis function
def run_analysis():
    """
    Processes Sentinel-2 imagery for the drawn polygon and date range.
    Creates an RGB composite and displays it on the map.
    """
    with status_output:
        status_output.clear_output()
        
        # Get the drawn geometry
        try:
            drawn_features = Map.draw_features
            if not drawn_features:
                print("❌ ERROR: Please draw a polygon on the map first!")
                return
            
            # Get the last drawn feature
            geometry = ee.Geometry(drawn_features[-1]['geometry'])
            
        except Exception as e:
            print(f"❌ ERROR: Could not get geometry. Please draw a polygon on the map first!")
            print(f"   Details: {str(e)}")
            return
        
        # Get dates
        start_date = start_date_picker.value.strftime('%Y-%m-%d')
        end_date = end_date_picker.value.strftime('%Y-%m-%d')
        
        print(f"🔵 Processing imagery from {start_date} to {end_date}...")
        
        # Load and filter Sentinel-2 data
        dataset = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                   .filterDate(start_date, end_date)
                   .filterBounds(geometry)
                   .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
                   .map(mask_s2_clouds))
        
        # Check if any images were found
        count = dataset.size().getInfo()
        if count == 0:
            print(f"⚠️  WARNING: No images found for this date range and location.")
            print(f"   Try expanding your date range or choosing a different time period.")
            return
        
        print(f"   Found {count} images with <5% cloud coverage")
        
        # Create composite
        composite = dataset.mean()
        clipped_composite = composite.clip(geometry)
        
        # Visualization parameters
        vis_params = {
            'bands': ['TCI_R', 'TCI_G', 'TCI_B'],
            'min': 0,
            'max': 0.3
        }
        
        # Add to map
        Map.addLayer(clipped_composite, vis_params, 'RGB Composite')
        
        print("✅ Composite added to map!")
        print("\nℹ️  Image Info:")
        print(f"   - Cloud coverage: <5%")
        print(f"   - Resolution: 10m")
        print(f"   - Bands: RGB (Natural Color)")
        print(f"   - Images used: {count}")
        
        # Store for export
        global export_image, export_geometry
        export_image = clipped_composite.select(['TCI_R', 'TCI_G', 'TCI_B'])
        export_geometry = geometry

# Run the analysis
run_analysis()

# Display status
display(status_output)

## Export Options

Choose how to export your composite:
1. **Local Download** - Download as GeoTIFF to your computer
2. **ArcGIS Online** - Upload directly to your ArcGIS Online account

### Option 1: Download to Local File

Download the composite as a GeoTIFF file to your local machine.

In [ ]:
# Download to local file
def download_local(output_folder='.'):
    """
    Downloads the RGB composite as a GeoTIFF to local folder.
    
    Args:
        output_folder: Folder path to save the file (default: current directory)
    """
    try:
        # Generate filename with date range
        start_str = start_date_picker.value.strftime('%Y%m%d')
        end_str = end_date_picker.value.strftime('%Y%m%d')
        filename = f'RGB_Sentinel2_{start_str}_to_{end_str}.tif'
        output_path = os.path.join(output_folder, filename)
        
        print(f"⏳ Downloading imagery to: {output_path}")
        print("   This may take several minutes depending on area size...")
        
        # Download using geemap
        geemap.ee_export_image(
            export_image,
            filename=output_path,
            scale=10,
            region=export_geometry,
            file_per_band=False
        )
        
        print(f"✅ Download complete: {output_path}")
        print(f"   You can now add this file to ArcGIS Pro or other GIS software.")
        
    except NameError:
        print("❌ ERROR: Please run the analysis first before downloading.")
    except Exception as e:
        print(f"❌ ERROR: Download failed - {str(e)}")

# Uncomment and modify the path below to download
# download_local(output_folder='C:/Users/YourName/Downloads')

### Option 2: Upload to ArcGIS Online

Upload the composite directly to your ArcGIS Online account as an imagery layer.

In [ ]:
# Import ArcGIS API for Python
try:
    from arcgis.gis import GIS
    from arcgis.raster import ImageryLayer
    import tempfile
    
    arcgis_available = True
except ImportError:
    print("⚠️  arcgis module not found. Install with: pip install arcgis")
    arcgis_available = False

In [ ]:
# Connect to ArcGIS Online
if arcgis_available:
    # Option 1: Use your ArcGIS Online credentials
    # gis = GIS('https://www.arcgis.com', 'your_username', 'your_password')
    
    # Option 2: Use interactive login (recommended)
    gis = GIS('home')  # This will use your ArcGIS Pro login if running in ArcGIS Pro
    
    print(f"✅ Connected to ArcGIS Online as: {gis.properties.user.username}")
else:
    print("❌ ArcGIS API not available. Please install: pip install arcgis")

In [ ]:
# Upload to ArcGIS Online
def upload_to_arcgis_online(title=None, tags=['sentinel-2', 'remote-sensing', 'quinhagak']):
    """
    Uploads the RGB composite to ArcGIS Online as an imagery layer.
    
    Args:
        title: Title for the layer (default: auto-generated from dates)
        tags: List of tags for the item
    """
    if not arcgis_available:
        print("❌ ERROR: ArcGIS API not available. Install with: pip install arcgis")
        return
    
    try:
        # Generate title and filename
        start_str = start_date_picker.value.strftime('%Y%m%d')
        end_str = end_date_picker.value.strftime('%Y%m%d')
        
        if title is None:
            title = f'Sentinel-2 RGB Composite {start_str} to {end_str}'
        
        print(f"⏳ Step 1/3: Downloading imagery from Earth Engine...")
        
        # Download to temporary file first
        with tempfile.TemporaryDirectory() as temp_dir:
            temp_file = os.path.join(temp_dir, f'sentinel2_{start_str}_{end_str}.tif')
            
            geemap.ee_export_image(
                export_image,
                filename=temp_file,
                scale=10,
                region=export_geometry,
                file_per_band=False
            )
            
            print(f"⏳ Step 2/3: Uploading to ArcGIS Online...")
            
            # Upload to ArcGIS Online
            item_properties = {
                'title': title,
                'tags': ','.join(tags),
                'description': f'Sentinel-2 RGB composite from {start_date_picker.value.strftime("%Y-%m-%d")} to {end_date_picker.value.strftime("%Y-%m-%d")}. '
                              f'Resolution: 10m. Cloud coverage: <5%. Processing: Earth Engine cloud masking applied.'
            }
            
            imagery_item = gis.content.add(item_properties, data=temp_file)
            
            print(f"⏳ Step 3/3: Publishing imagery layer...")
            
            # Publish as imagery layer
            published_item = imagery_item.publish()
            
            print(f"\n✅ Upload complete!")
            print(f"   Title: {title}")
            print(f"   Item ID: {published_item.id}")
            print(f"   URL: {published_item.homepage}")
            print(f"\n   You can now add this layer to your web maps and ArcGIS Pro projects.")
            
            return published_item
        
    except NameError:
        print("❌ ERROR: Please run the analysis first before uploading.")
    except Exception as e:
        print(f"❌ ERROR: Upload failed - {str(e)}")

# Uncomment the line below to upload to ArcGIS Online
# published_layer = upload_to_arcgis_online()

## Additional Information

### About Sentinel-2
- **Mission**: European Space Agency (ESA) Earth observation mission
- **Temporal Resolution**: 5-day revisit time at the equator
- **Spatial Resolution**: 10m (RGB and NIR bands)
- **Coverage**: Global

### Bands Used
- **TCI_R**: True Color Image - Red (Band 4)
- **TCI_G**: True Color Image - Green (Band 3)
- **TCI_B**: True Color Image - Blue (Band 2)

### Tips
- Summer images (June-August) show more detail with less snow cover
- The script filters for images with <5% cloud coverage
- If no images are found, try expanding your date range
- Multiple images in the date range are averaged together

### Using in ArcGIS Pro
1. **Local File**: Use the download function, then add the GeoTIFF to your ArcGIS Pro map
2. **ArcGIS Online**: Upload directly, then access via your Portal in ArcGIS Pro
3. The imagery layer will appear in your Content and can be shared with your organization